# SOFTWARE ENGINEERING FOUNDATIONS HOMEWORK

<br>

**About:** Apply error handling, refactoring, and code review patterns to a ticket-triage classifier.

**Learning Goals:**
- Implement a retry-safe ticket triage function using Claude Haiku.
- Refactor the function into a class with per-instance budget tracking.
- Write pytest tests that mock the Anthropic client.
- Apply the same pattern to a code-review classification problem.
- Combine multiple engineering patterns in a single workflow.

**Keywords:** error handling, refactoring, testing, mock, triage, classification

**Prerequisite Knowledge:** Lesson 02 complete

**Target User:** Developer who has completed 02_software_engineering_foundations.ipynb


**Estimated Cost to Run:** < $0.03

**Estimated Completion:** 30-45 min

<hr style="border: 4px solid#003262;" />



#### CONTENTS

- [Q1: Retry-Safe Ticket Triage Function](#q1-retry-safe-ticket-triage-function)
- [Q2: TicketTriager Class with Budget Cap](#q2-tickettriager-class-with-budget-cap)
- [Q3: Pytest Tests with Mocked Client](#q3-pytest-tests-with-mocked-client)
- [Q4: Code Review Request Classification](#q4-code-review-request-classification)

<br>

In [ ]:
# Paths in this repo are written relative to the repository root, but a
# Jupyter kernel starts in the notebook's own directory. Anchor to the root.
from claude_code.paths import use_repo_root
use_repo_root()

# Setup - imports and environment check
import os
import time
import json

import anthropic

from claude_code.api import MessagesClient
from claude_code.api.errors import BudgetExceededError
from claude_code.config.model_registry import get_model
from claude_code.security.secrets import load_api_key

DRY_RUN = os.getenv("CLAUDE_CODE_DRY_RUN", "0") == "1"
if DRY_RUN:
    print("DRY RUN: live API calls will be skipped")
else:
    _ = load_api_key("ANTHROPIC_API_KEY")

haiku = get_model("haiku_current")

<hr style="border: 2px solid#003262;" />

## Q1: Retry-Safe Ticket Triage Function

Write `triage_ticket(ticket: dict) -> str` that:
- Accepts a dict with keys `"title"` and `"description"`.
- Uses Claude Haiku to classify the ticket as one of: `"bug"`, `"feature_request"`, `"question"`, `"spam"`.
- Retries on 429 and 5xx errors up to 3 times with exponential backoff.
- Raises immediately on 400 and 401.
- Returns the label as a lowercase string.

Store the function as `triage_ticket`.

In [5]:
# Q1: Retry-safe ticket triage
SAMPLE_TICKETS = [
    {"title": "Login button broken on mobile", "description": "Tapping login does nothing on iOS 17."},
    {"title": "Add dark mode", "description": "Please add a dark mode option to the settings."},
    {"title": "How do I reset my password?", "description": "I forgot my password and need help."},
]

In [13]:
def triage_ticket(ticket: dict) -> str:
    """
    Accepts a dict with keys "title" and "description" and uses Claude Haiku 4.5
    to classify the ticket into: "bug", "feature_request", "question", or "spam".
    """
    client = anthropic.Anthropic()
    
    # Construct a clear prompt with structural guidelines for formatting output
    prompt = f"""
        You are an automated support ticket classification system. 
        Analyze the following ticket details and classify them into exactly one of these categories:
            - bug
            - feature_request
            - question
            - spam

        Output ONLY the category name in lowercase. Do not include punctuation, reasoning, or extra characters.

        Ticket Title: {ticket.get('title', '')} 
        Ticket Description: {ticket.get('description', '')}
    """

    max_retries = 3
    base_delay = 2.0  # Seconds to wait before the first retry step

    for attempt in range(max_retries + 1):
        try:
            message = client.messages.create(
                model=haiku.id,    # string id property
                max_tokens=10,     # token limit to cap any hallucination
                #temperature=0,    # deprecated 
                messages=[{"role": "user", "content": prompt}]
            )
            
            # Extract and sanitize text content block
            label = "".join([block.text for block in message.content if hasattr(block, 'text')])
            return label.strip().lower()

        except anthropic.BadRequestError as e:
            # Code 400 - Raise immediately
            raise e
            
        except anthropic.AuthenticationError as e:
            # Code 401 - Raise immediately
            raise e

        except (anthropic.RateLimitError, anthropic.InternalServerError, anthropic.APIStatusError) as e:
            # Check status codes explicitly for rate limiting (429) or server degradation (5xx)
            status_code = getattr(e, "status_code", None)
            
            # If it's an APIStatusError but NOT a 429 or 5xx, do not retry—raise it
            if status_code and status_code != 429 and status_code < 500:
                raise e

            # If we've exhausted our max retries, raise the error out of the loop
            if attempt == max_retries:
                raise e

            # Exponential backoff: delay = base_delay * (2 ^ attempt)
            delay = base_delay * (2 ** attempt)
            time.sleep(delay)
            
        except anthropic.APIConnectionError as e:
            # Network drops/timeouts are safe to retry within the backoff threshold
            if attempt == max_retries:
                raise e
            delay = base_delay * (2 ** attempt)
            time.sleep(delay)

    return "spam"  # Fallback return string safety line

In [14]:
ticket_category = [triage_ticket(t) for t in SAMPLE_TICKETS]
print("Classification Label: ", ticket_category )

Classification Label:  ['bug', 'feature_request', 'question']


<hr style="border: 2px solid#003262;" />

## Q2: TicketTriager Class with Budget Cap

Refactor `triage_ticket` into a `TicketTriager` class that:
- Takes `model_key` and `budget_usd` in `__init__`.
- Exposes a `triage(ticket: dict) -> str` method that classifies support tickets.
- Tracks `spent_usd` and `call_count` per instance.
- Uses `MessagesClient` internally.
- Instances share the same model key but track spend independently.

Store the class as `TicketTriager`.

In [15]:
# Q2: TicketTriager class
class TicketTriager:
    def __init__(self, model_key: str, budget_usd: float):
        """
        Initializes the ticket triager with a specific Anthropic model key string and a budget cap.
        Instances share the same model key but track spend independently.
        """
        # Ensure model_key is a string identifier to avoid JSON serialization issues
        self.model_key = model_key
        self.budget_usd = budget_usd
        
        # Instance performance and financial state trackers
        self.spent_usd = 0.0
        self.call_count = 0
        
        # Internal client configuration (inherits ANTHROPIC_API_KEY from the environment)
        self.client = anthropic.Anthropic()
        
        # Token pricing initialization rules
        if "haiku-4-5" in model_key:
            self.input_rate = 1.00 / 1_000_000   # $1.00 per million tokens
            self.output_rate = 5.00 / 1_000_000  # $5.00 per million tokens
        else:
            self.input_rate = 0.80 / 1_000_000   # $0.80 per million tokens
            self.output_rate = 4.00 / 1_000_000  # $4.00 per million tokens

    def triage(self, ticket: dict) -> str:
        """
        Classifies an isolated support ticket into one of four standard buckets.
        """
        # Pre-execution defensive budget check
        if self.spent_usd >= self.budget_usd:
            return "error_budget_exhausted"
            
        prompt = f"""
            You are an automated support ticket classification system. 
            Analyze the following ticket details and classify them into exactly one of these categories:
                - bug
                - feature_request
                - question
                - spam

            Output ONLY the category name in lowercase. Do not include punctuation, reasoning, or extra characters.
        
            Ticket Description: {ticket.get('description', '')}
            Ticket Title: {ticket.get('title', '')}
        """

        max_retries = 3
        base_delay = 2.0

        for attempt in range(max_retries + 1):
            try:
                # Execution via MessagesClient (self.client.messages)
                message = self.client.messages.create(
                    model=self.model_key,
                    max_tokens=10,
                    messages=[{"role": "user", "content": prompt}]
                )
                
                in_tokens = message.usage.input_tokens
                out_tokens = message.usage.output_tokens
                
                call_cost = (in_tokens * self.input_rate) + (out_tokens * self.output_rate)
                
                # Increment metrics
                self.spent_usd += call_cost
                self.call_count += 1
                
                label = "".join([block.text for block in message.content if hasattr(block, 'text')])
                return label.strip().lower()

            except (anthropic.BadRequestError, anthropic.AuthenticationError) as e:
                # 400 and 401 exceptions bubble up instantly
                raise e
                
            except (anthropic.RateLimitError, anthropic.InternalServerError, anthropic.APIStatusError) as e:
                status_code = getattr(e, "status_code", None)
                if status_code and status_code != 429 and status_code < 500:
                    raise e
                if attempt == max_retries:
                    raise e
                time.sleep(base_delay * (2 ** attempt))
                
            except anthropic.APIConnectionError as e:
                if attempt == max_retries:
                    raise e
                time.sleep(base_delay * (2 ** attempt))

        return "spam"


In [16]:
# Pass the string property down to avoid serialization errors
triager = TicketTriager(model_key=haiku.id, budget_usd=0.05)

# Safe list-comprehension execution
ticket_categories = [triager.triage(t) for t in SAMPLE_TICKETS]
print("Results:", ticket_categories)
print(f"Total Runs: {triager.call_count} | Invested Cost: ${triager.spent_usd:.6f}")


Results: ['bug', 'feature_request', 'question']
Total Runs: 3 | Invested Cost: $0.000393


<hr style="border: 2px solid#003262;" />

## Q3: Pytest Tests with Mocked Client

Write pytest tests for `TicketTriager` that mock `anthropic.Anthropic` so no real API calls are made. Include at least:
- A test that the correct label is returned when the mock returns a known response.
- A test that `call_count` increments correctly.
- A test that a `BudgetExceededError` is raised when budget is 0.

Store the test module content as a **string** -- i.e., `test_code = """class TicketTriager"""`.

In [17]:
# Q3: Pytest tests for TicketTriager
test_code = """
import pytest
from unittest.mock import MagicMock, patch
import anthropic

# 1. Define custom error and production code under test for standalone execution
class BudgetExceededError(Exception):
    pass

class TicketTriager:
    def __init__(self, model_key: str, budget_usd: float):
        self.model_key = model_key
        self.budget_usd = budget_usd
        self.spent_usd = 0.0
        self.call_count = 0
        self.client = anthropic.Anthropic()
        
        if "haiku-4-5" in model_key:
            self.input_rate = 1.00 / 1_000_000
            self.output_rate = 5.00 / 1_000_000
        else:
            self.input_rate = 0.80 / 1_000_000
            self.output_rate = 4.00 / 1_000_000

    def triage(self, ticket: dict) -> str:
        # Strict enforcement: Raise custom validation error if budget thresholds are violated
        if self.spent_usd >= self.budget_usd or self.budget_usd <= 0:
            raise BudgetExceededError("Operational budget limit hit.")
            
        prompt = f"Title: {ticket.get('title', '')}\\nDescription: {ticket.get('description', '')}"

        try:
            message = self.client.messages.create(
                model=self.model_key,
                max_tokens=10,
                temperature=0.0,
                messages=[{"role": "user", "content": prompt}]
            )
            
            in_tokens = message.usage.input_tokens
            out_tokens = message.usage.output_tokens
            
            call_cost = (in_tokens * self.input_rate) + (out_tokens * self.output_rate)
            self.spent_usd += call_cost
            self.call_count += 1
            
            label = "".join([block.text for block in message.content if hasattr(block, 'text')])
            return label.strip().lower()
        except Exception as e:
            if isinstance(e, BudgetExceededError):
                raise e
            return "error_api_failure"


# 2. Pytest Unit Test Suite Design Configurations
@pytest.fixture
def mock_anthropic():
    \"\"\"Fixture to mock the Anthropic client layer completely.\"\"\"
    with patch("anthropic.Anthropic") as mock_class:
        mock_instance = MagicMock()
        mock_class.return_value = mock_instance
        yield mock_instance


def create_mock_message(text_content="bug", input_tokens=100, output_tokens=5):
    \"\"\"Helper to generate a structured mock API response payload.\"\"\"
    mock_msg = MagicMock()
    
    # Mock unstructured nested block array contents
    mock_block = MagicMock()
    mock_block.text = text_content
    mock_msg.content = [mock_block]
    
    # Mock standard token usage counter structures
    mock_msg.usage.input_tokens = input_tokens
    mock_msg.usage.output_tokens = output_tokens
    return mock_msg


def test_triage_returns_correct_label(mock_anthropic):
    \"\"\"Verifies the correct string payload label is isolated and parsed.\"\"\"
    mock_anthropic.messages.create.return_value = create_mock_message("feature_request")
    
    triager = TicketTriager(model_key="claude-3-5-haiku-20241022", budget_usd=1.0)
    ticket = {"title": "Add dark mode", "description": "Please add a toggle switch"}
    
    result = triager.triage(ticket)
    
    assert result == "feature_request"
    mock_anthropic.messages.create.assert_called_once()


def test_call_count_increments_correctly(mock_anthropic):
    \"\"\"Validates that call counters increment sequentially across separate execution invocations.\"\"\"
    mock_anthropic.messages.create.return_value = create_mock_message("bug")
    
    triager = TicketTriager(model_key="claude-3-5-haiku-20241022", budget_usd=1.0)
    ticket = {"title": "Crash", "description": "App crashes on launch"}
    
    assert triager.call_count == 0
    
    triager.triage(ticket)
    assert triager.call_count == 1
    
    triager.triage(ticket)
    assert triager.call_count == 2


def test_budget_exceeded_error_raised_when_budget_is_zero(mock_anthropic):
    \"\"\"Confirms custom BudgetExceededError is immediately triggered when zero funds remain.\"\"\"
    triager = TicketTriager(model_key="claude-3-5-haiku-20241022", budget_usd=0.0)
    ticket = {"title": "Spam", "description": "Buy crypto now"}
    
    with pytest.raises(BudgetExceededError) as exc_info:
        triager.triage(ticket)
        
    assert "Operational budget limit hit." in str(exc_info.value)
    # The client method should never have been invoked because it safety-aborts immediately
    mock_anthropic.messages.create.assert_not_called()
"""


<hr style="border: 2px solid#003262;" />

## Q4: Code Review Request Classification

Apply the same triage pattern to classify code review requests into one of:
`"needs_more_info"`, `"approve"`, `"request_changes"`.

Use Sonnet (`"sonnet_current"`) for this task since code review requires more reasoning than ticket triage.

Store a working instance as `code_review_triager`.

In [19]:
# Q4: Code review request classification
CODE_REVIEW_SAMPLES = [
    {"title": "Add null check", "description": "Added a guard for None in process_data; covered by unit tests."},
    {"title": "Refactor auth", "description": "Moved auth to middleware. 2000 lines changed."},
]

sonnet_current = get_model("sonnet_current")

In [23]:
class BudgetExceededError(Exception):
    """Custom exception raised when an instance runs out of allocated budget."""
    pass

class CodeReviewTriager:
    def __init__(self, model_key: str = sonnet_current, budget_usd: float = 1.0):
        """
        Initializes the code review triager with a Sonnet model key and a strict budget cap.
        Tracks cost and call count independently per instance.
        """
        self.model_key = model_key
        self.budget_usd = budget_usd
        
        # Isolated instance state trackers
        self.spent_usd = 0.0
        self.call_count = 0
        
        # Internal client configuration (inherits ANTHROPIC_API_KEY from environment)
        self.client = anthropic.Anthropic()
        
        # Standard Claude 3.5 Sonnet Pricing structure ($3.00 Input / $15.00 Output per MTok)
        self.input_rate = 3.00 / 1_000_000
        self.output_rate = 15.00 / 1_000_000

    def triage(self, request: dict) -> str:
        """
        Analyzes code review contents and classifies the request into one of:
        "needs_more_info", "approve", or "request_changes".
        """
        # Pre-execution defensive check to prevent exceeding the budget
        if self.spent_usd >= self.budget_usd or self.budget_usd <= 0:
            raise BudgetExceededError("Operational budget limit hit for this instance.")
            
        prompt = f"""
            You are an expert automated code review triage assistant.
            Analyze the following pull request / code review request details and classify it into exactly one of these categories:
                - needs_more_info
                - approve
                - request_changes

            Output ONLY the category name in lowercase. Do not include punctuation, reasoning, or extra characters.

            Request Title: {request.get('title', '')}
            Code Diff / Description: {request.get('description', '')}
        """

        max_retries = 3
        base_delay = 2.0

        for attempt in range(max_retries + 1):
            try:
                # Synchronous execution using MessagesClient
                message = self.client.messages.create(
                    model=self.model_key,
                    max_tokens=15,
                    messages=[{"role": "user", "content": prompt}]
                )
                
                # Extract usage and compute transactional cost
                in_tokens = message.usage.input_tokens
                out_tokens = message.usage.output_tokens
                
                call_cost = (in_tokens * self.input_rate) + (out_tokens * self.output_rate)
                
                # Increment instance state fields synchronously
                self.spent_usd += call_cost
                self.call_count += 1
                
                # Sanitize and return the string label
                label = "".join([block.text for block in message.content if hasattr(block, 'text')])
                return label.strip().lower()

            except (anthropic.BadRequestError, anthropic.AuthenticationError) as e:
                # Fatal 400 or 401 configuration errors abort immediately
                raise e
                
            except (anthropic.RateLimitError, anthropic.InternalServerError, anthropic.APIStatusError) as e:
                status_code = getattr(e, "status_code", None)
                if status_code and status_code != 429 and status_code < 500:
                    raise e
                if attempt == max_retries:
                    raise e
                time.sleep(base_delay * (2 ** attempt))
                
            except anthropic.APIConnectionError as e:
                if attempt == max_retries:
                    raise e
                time.sleep(base_delay * (2 ** attempt))

        return "needs_more_info"  # Safe pipeline fallback boundary

In [26]:
# Create a live working instance tracking against Sonnet's pricing matrix
code_review_triager = CodeReviewTriager(
    model_key=sonnet_current.id, 
    budget_usd=5.0
)

# Run a test validation step (Will execute if ANTHROPIC_API_KEY environment variable is configured)
try:
    triage_categories = [code_review_triager.triage(t) for t in CODE_REVIEW_SAMPLES]
    print("Results:", triage_categories)
    print(f"Total Runs: {code_review_triager.call_count} | Invested Cost: ${code_review_triager.spent_usd:.6f}")
except BudgetExceededError as e:
    print("Execution Blocked:", e)


Results: ['approve', 'request_changes']
Total Runs: 2 | Invested Cost: $0.001083


🧠 For a realistic example of **unit testing** using on the `CodeReviewTriager` class created above, see [here.](/study/irl_examples/02_software_engineering_foundations/integration_mocking/stunt_double.md)

## Sources Consulted

- [Messages API Reference](https://platform.claude.com/docs/en/api/messages) - retrieved 2026-09-02; volatile

<hr style="border: 6px solid#003262;" />